# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline


In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-mini"
openai = OpenAI()

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


OpenAI API Key exists and begins sk-proj-


In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.


In [15]:
# Let's start by making a useful function

ticket_prices = {
    "london": "$799",
    "paris": "$899",
    "tokyo": "$1400",
    "berlin": "$499",
    "singapore": "$999",
}


def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [16]:
get_ticket_price("singapore")

Tool called for city singapore


'The price of a ticket to singapore is $999'

In [8]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            # destination_city is the input (parameter) of the function
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False,
    },
}

In [10]:
# And this is included in a list of tools (wrapped in another json):

tools = [{"type": "function", "function": price_function}]

In [11]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:


In [18]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,  # 1. we need to pass in the TOOLS aka the json that we wrote just now
    )

    # 2. we need to check if the LLM requires us to call a tool, and call the tool if neede
    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(
            message
        )  # put in the conversation history the message from open AI to call the tool
        messages.append(response)  # and then add the answer from the tool

        for message in messages:
            print(message)

        response = openai.chat.completions.create(
            model=MODEL, messages=messages
        )  # send the entire conversation history + openAI tool call + answer from tool call into the LLM

    return response.choices[0].message.content

In [ ]:
# We have to write that function handle_tool_call:


def handle_tool_call(message):
    tool_call = message.tool_calls[
        0
    ]  # problem: you are just taking the first tool call, we do not support if the ai wants to make multiple tool calls, like if the user says they wish to go to berlin and paris -> ERROR!
    if (
        tool_call.function.name == "get_ticket_price"
    ):  # we look at the object in tool_call
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get("destination_city")
        price_details = get_ticket_price(
            city
        )  # this is just calling the function that we wrote previoiusly!
        response = {
            "role": "tool",  # NEW ROLE! (previously we have 'user' and 'assistant')
            "content": price_details,
            "tool_call_id": tool_call.id,  # this is a MUST to have
        }
    return response

In [19]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


Tool called for city Berlin
{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}
{'role': 'user', 'content': 'hi there'}
{'role': 'assistant', 'content': 'Hello! How can I assist you with your flight today?'}
{'role': 'user', 'content': 'i want to go to berlin'}
{'role': 'assistant', 'content': 'Would you like to know the price of a return ticket to Berlin?'}
{'role': 'user', 'content': 'yes'}
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_uRXo82wZlbpXrm0DXUXDgpnb', function=Function(arguments='{"destination_city":"Berlin"}', name='get_ticket_price'), type='function')])
{'role': 'tool', 'content': 'The price of a ticket to Berlin is $499', 'tool_call_id': 'call_uRXo82wZlbpXrm0DXUXDgpnb'}


## Let's make a couple of improvements

Handling multiple tool calls in 1 response (eg. message: I want to get the ticket prices to Berlin and Paris)

Handling multiple tool calls 1 after another (eg. message: Please get the ticket prices to London, and ONLY if it's under $1000, get the ticket prices to Paris.)


In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )
    response = openai.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )

    if (
        response.choices[0].finish_reason == "tool_calls"
    ):  # handling multiple tool calls in 1 response
        message = response.choices[0].message
        responses = handle_tool_calls(
            message
        )  # another function, handle multiple tool call
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)

    return response.choices[0].message.content

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")
            price_details = get_ticket_price(
                city
            )  # improvement: call the function directly if the name matches a function name (instead of using multiple if-statements), else handle the error gracefully as well if the tool_call.function.name does not match the name of an existing function just to be safe.
            responses.append(
                {"role": "tool", "content": price_details, "tool_call_id": tool_call.id}
            )
    return responses

In [28]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


Tool called for city london
Tool called for city singapore
{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}
{'role': 'user', 'content': 'get ticket prices to london, and check ONLY if it is more than 500 dollars, get the ticket prices for singapore'}
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Gycv89ksy4KjdSilCNlBgf9Q', function=Function(arguments='{"destination_city": "london"}', name='get_ticket_price'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_SaNAhqYAdlRAXcCPaJG0GW2L', function=Function(arguments='{"destination_city": "singapore"}', name='get_ticket_price'), type='function')])
{'role': 'tool', 'content': 'The price of a ticket to london is $799', 'tool_call_id': 'call_

In [29]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )
    response = openai.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )

    while (
        response.choices[0].finish_reason == "tool_calls"
    ):  # note the previous version is 'if', just change it to 'while' to allow it to loop through all the tool calls. -> handling multiple tool calls one after another. continually call the tools while it needs the tool calls. (unlikely to go on forever, LLMs don't really do that forever...)
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)

        for message in messages:
            print(message)

        response = openai.chat.completions.create(
            model=MODEL, messages=messages, tools=tools
        )

    return response.choices[0].message.content

#### Let it fetch data from a database instead of a python dictionary, can connect to Postgres or set up SQL in a managed service like supabase.


In [26]:
import sqlite3


In [31]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute(
        "CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)"  # SQL query
    )
    conn.commit()

In [32]:
def get_ticket_price(city):
    print(
        f"DATABASE TOOL CALLED: Getting price for {city}", flush=True
    )  # flush=true means print the statement ones immediately without waiting.
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "SELECT price FROM prices WHERE city = ?", (city.lower(),)
        )  # we are just appending the city name so we don't get SQL injection attacks
        result = cursor.fetchone()
        return (
            f"Ticket price to {city} is ${result[0]}"
            if result
            else "No price data available for this city"
        )

In [33]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'No price data available for this city'

In [34]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?",
            (city.lower(), price, price),
        )
        conn.commit()

In [36]:
ticket_prices = {
    "london": 799,
    "paris": 899,
    "tokyo": 1420,
    "sydney": 2999,
    "singapore": 999,
}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [37]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420.0'

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for Sydney
{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}
{'role': 'user', 'content': "hi there, i want to go to sydney, what's the ticket price?"}
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_ZXukZqrC5WkO0gkCR8CbPop6', function=Function(arguments='{"destination_city":"Sydney"}', name='get_ticket_price'), type='function')])
{'role': 'tool', 'content': 'Ticket price to Sydney is $2999.0', 'tool_call_id': 'call_ZXukZqrC5WkO0gkCR8CbPop6'}
DATABASE TOOL CALLED: Getting price for Singapore
DATABASE TOOL CALLED: Getting price for London
{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no mor

## Exercise

Add a tool to set the price of a ticket!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Hopefully this hardly needs to be stated! You now have the ability to give actions to your LLMs. This Airline Assistant can now do more than answer questions - it could interact with booking APIs to make bookings!</span>
        </td>
    </tr>
</table>
